## Dataset 1

In [3]:
"""
Analisis Statistik: Wilcoxon Signed-Rank Test + Confidence Interval
Perbandingan Proposed (LightGBM P=60%, 129 fitur) vs Baseline (Old)
Metrik: Accuracy, F1 Macro, F1 Weighted
"""

import numpy as np
from scipy import stats

# ─────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────

baseline = {
    "Accuracy":    np.array([0.9859, 0.9892, 0.9871, 0.9879, 0.9842]),
    "F1 Macro":    np.array([0.9848, 0.9883, 0.9862, 0.9870, 0.9830]),
    "F1 Weighted": np.array([0.9858, 0.9892, 0.9871, 0.9879, 0.9842]),
}

proposed = {
    "Accuracy":    np.array([0.9896, 0.9900, 0.9900, 0.9900, 0.9908]),
    "F1 Macro":    np.array([0.9888, 0.9893, 0.9893, 0.9893, 0.9902]),
    "F1 Weighted": np.array([0.9896, 0.9900, 0.9900, 0.9900, 0.9908]),
}

METRICS = ["Accuracy", "F1 Macro", "F1 Weighted"]
ALPHA   = 0.05

# ─────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────

def bootstrap_ci(diff, n_boot=10_000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    means = [rng.choice(diff, size=len(diff), replace=True).mean()
             for _ in range(n_boot)]
    return np.percentile(means, (100-ci)/2), np.percentile(means, 100-(100-ci)/2)

def rank_biserial(diff):
    pos = np.sum(diff[diff > 0])
    neg = abs(np.sum(diff[diff < 0]))
    return (pos - neg) / (pos + neg) if (pos + neg) > 0 else 0.0

def effect_label(r):
    r = abs(r)
    if r < 0.1:   return "sangat kecil"
    elif r < 0.3: return "kecil"
    elif r < 0.5: return "sedang"
    else:         return "besar"

SEP  = "=" * 70
SEP2 = "-" * 70

print(SEP)
print("  WILCOXON SIGNED-RANK TEST + BOOTSTRAP CI")
print("  Proposed : LightGBM P=60% (129 fitur)")
print("  Baseline : Old model")
print(f"  Folds: 5  |  α = {ALPHA}  |  Bootstrap CI: 95%  |  n_boot: 10.000")
print(SEP)

results = {}

for metric in METRICS:
    b    = baseline[metric]
    p    = proposed[metric]
    diff = p - b

    mean_b    = b.mean()
    mean_p    = p.mean()
    mean_diff = diff.mean()

    try:
        stat, p_val = stats.wilcoxon(p, b, alternative="two-sided")
    except ValueError as e:
        stat, p_val = np.nan, np.nan

    ci_lo, ci_hi = bootstrap_ci(diff)
    r = rank_biserial(diff)

    results[metric] = dict(mean_b=mean_b, mean_p=mean_p,
                           mean_diff=mean_diff, stat=stat,
                           p_val=p_val, ci_lo=ci_lo, ci_hi=ci_hi, r=r)

    sig  = (not np.isnan(p_val)) and p_val < ALPHA
    mark = "✓ SIGNIFIKAN" if sig else "✗ tidak signifikan"
    ci_covers_zero = ci_lo <= 0 <= ci_hi

    print(f"\n  Metrik  : {metric}")
    print(SEP2)
    print(f"  Baseline mean ± std  : {mean_b:.4f} ± {b.std(ddof=1):.4f}")
    print(f"  Proposed mean ± std  : {mean_p:.4f} ± {p.std(ddof=1):.4f}")
    print(f"  Δ mean (Prop - Base) : {mean_diff:+.4f}")
    print(f"  Bootstrap 95% CI (Δ) : [{ci_lo:+.4f},  {ci_hi:+.4f}]"
          + ("  ← CI mencakup 0" if ci_covers_zero else "  ← CI tidak mencakup 0 ✓"))
    print(f"  Wilcoxon W           : {stat:.1f}")
    print(f"  p-value              : {p_val:.4f}")
    print(f"  Kesimpulan           : {mark}")
    print(f"  Effect size r        : {r:+.4f}  → {effect_label(r)}")

# ─────────────────────────────────────────────
# TABEL RINGKASAN
# ─────────────────────────────────────────────

print(f"\n\n{SEP}")
print("  TABEL RINGKASAN")
print(SEP)
print(f"  {'Metrik':<14} {'Base':>7} {'Prop':>7} {'Δ':>8}  {'95% CI Δ':>22}  {'p-val':>7}  {'Sig':>4}  {'r':>7}  Efek")
print(SEP2)

for m, v in results.items():
    ci_str = f"[{v['ci_lo']:+.4f}, {v['ci_hi']:+.4f}]"
    sig    = (not np.isnan(v['p_val'])) and v['p_val'] < ALPHA
    s_str  = "YA ✓" if sig else "tidak"
    p_str  = f"{v['p_val']:.4f}" if not np.isnan(v['p_val']) else "  NaN"
    print(f"  {m:<14} {v['mean_b']:>7.4f} {v['mean_p']:>7.4f} {v['mean_diff']:>+8.4f}  {ci_str:>22}  {p_str:>7}  {s_str:>4}  {v['r']:>+7.3f}  {effect_label(v['r'])}")

print(SEP)
print("""
  CATATAN
  • Wilcoxon two-sided, α=0.05. Dengan 5 fold, power uji rendah →
    perhatikan CI dan effect size sebagai bukti tambahan.
  • CI tidak mencakup 0  → perbedaan konsisten di semua fold.
  • |r| ≥ 0.5 = effect size besar meski p belum < 0.05.
""")

  WILCOXON SIGNED-RANK TEST + BOOTSTRAP CI
  Proposed : LightGBM P=60% (129 fitur)
  Baseline : Old model
  Folds: 5  |  α = 0.05  |  Bootstrap CI: 95%  |  n_boot: 10.000

  Metrik  : Accuracy
----------------------------------------------------------------------
  Baseline mean ± std  : 0.9869 ± 0.0019
  Proposed mean ± std  : 0.9901 ± 0.0004
  Δ mean (Prop - Base) : +0.0032
  Bootstrap 95% CI (Δ) : [+0.0016,  +0.0050]  ← CI tidak mencakup 0 ✓
  Wilcoxon W           : 0.0
  p-value              : 0.0625
  Kesimpulan           : ✗ tidak signifikan
  Effect size r        : +1.0000  → besar

  Metrik  : F1 Macro
----------------------------------------------------------------------
  Baseline mean ± std  : 0.9859 ± 0.0020
  Proposed mean ± std  : 0.9894 ± 0.0005
  Δ mean (Prop - Base) : +0.0035
  Bootstrap 95% CI (Δ) : [+0.0019,  +0.0054]  ← CI tidak mencakup 0 ✓
  Wilcoxon W           : 0.0
  p-value              : 0.0625
  Kesimpulan           : ✗ tidak signifikan
  Effect size r      

## Dataset 2

In [4]:
"""
Analisis Statistik: Wilcoxon Signed-Rank Test + Bootstrap CI
Proposed: RandomForest P=40% (76 fitur) vs Baseline
Metrik: Accuracy, F1 Macro, F1 Weighted
"""

import numpy as np
from scipy import stats

baseline = {
    "Accuracy":    np.array([0.9630, 0.9618, 0.9629, 0.9629, 0.9623]),
    "F1 Macro":    np.array([0.9622, 0.9609, 0.9621, 0.9621, 0.9615]),
    "F1 Weighted": np.array([0.9630, 0.9618, 0.9629, 0.9629, 0.9623]),
}

proposed = {
    "Accuracy":    np.array([0.9614, 0.9611, 0.9618, 0.9625, 0.9618]),
    "F1 Macro":    np.array([0.9606, 0.9602, 0.9610, 0.9617, 0.9610]),
    "F1 Weighted": np.array([0.9614, 0.9611, 0.9618, 0.9625, 0.9618]),
}

METRICS = ["Accuracy", "F1 Macro", "F1 Weighted"]
ALPHA   = 0.05

def bootstrap_ci(diff, n_boot=10_000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    means = [rng.choice(diff, size=len(diff), replace=True).mean()
             for _ in range(n_boot)]
    return np.percentile(means, (100-ci)/2), np.percentile(means, 100-(100-ci)/2)

def rank_biserial(diff):
    pos = np.sum(diff[diff > 0])
    neg = abs(np.sum(diff[diff < 0]))
    return (pos - neg) / (pos + neg) if (pos + neg) > 0 else 0.0

def effect_label(r):
    r = abs(r)
    if r < 0.1:   return "sangat kecil"
    elif r < 0.3: return "kecil"
    elif r < 0.5: return "sedang"
    else:         return "besar"

SEP  = "=" * 70
SEP2 = "-" * 70

print(SEP)
print("  WILCOXON SIGNED-RANK TEST + BOOTSTRAP CI")
print("  Proposed : RandomForest P=40% (76 fitur)")
print("  Baseline : Original model")
print(f"  Folds: 5  |  α = {ALPHA}  |  Bootstrap CI: 95%  |  n_boot: 10.000")
print(SEP)

results = {}

for metric in METRICS:
    b    = baseline[metric]
    p    = proposed[metric]
    diff = p - b

    mean_b    = b.mean()
    mean_p    = p.mean()
    mean_diff = diff.mean()

    try:
        stat, p_val = stats.wilcoxon(p, b, alternative="two-sided")
    except ValueError:
        stat, p_val = np.nan, np.nan

    ci_lo, ci_hi = bootstrap_ci(diff)
    r = rank_biserial(diff)

    results[metric] = dict(mean_b=mean_b, mean_p=mean_p,
                           mean_diff=mean_diff, stat=stat,
                           p_val=p_val, ci_lo=ci_lo, ci_hi=ci_hi, r=r)

    sig  = (not np.isnan(p_val)) and p_val < ALPHA
    mark = "✓ SIGNIFIKAN" if sig else "✗ tidak signifikan"
    ci_covers_zero = ci_lo <= 0 <= ci_hi

    print(f"\n  Metrik  : {metric}")
    print(SEP2)
    print(f"  Baseline mean ± std  : {mean_b:.4f} ± {b.std(ddof=1):.4f}")
    print(f"  Proposed mean ± std  : {mean_p:.4f} ± {p.std(ddof=1):.4f}")
    print(f"  Δ mean (Prop - Base) : {mean_diff:+.4f}")
    print(f"  Bootstrap 95% CI (Δ) : [{ci_lo:+.4f},  {ci_hi:+.4f}]"
          + ("  ← CI mencakup 0" if ci_covers_zero else "  ← CI tidak mencakup 0 ✓"))
    print(f"  Wilcoxon W           : {stat:.1f}")
    print(f"  p-value              : {p_val:.4f}")
    print(f"  Kesimpulan           : {mark}")
    print(f"  Effect size r        : {r:+.4f}  → {effect_label(r)}")

print(f"\n\n{SEP}")
print("  TABEL RINGKASAN")
print(SEP)
print(f"  {'Metrik':<14} {'Base':>7} {'Prop':>7} {'Δ':>8}  {'95% CI Δ':>22}  {'p-val':>7}  {'Sig':>6}  {'r':>7}  Efek")
print(SEP2)

for m, v in results.items():
    ci_str = f"[{v['ci_lo']:+.4f}, {v['ci_hi']:+.4f}]"
    sig    = (not np.isnan(v['p_val'])) and v['p_val'] < ALPHA
    s_str  = "YA ✓" if sig else "tidak"
    p_str  = f"{v['p_val']:.4f}" if not np.isnan(v['p_val']) else "  NaN"
    print(f"  {m:<14} {v['mean_b']:>7.4f} {v['mean_p']:>7.4f} {v['mean_diff']:>+8.4f}  {ci_str:>22}  {p_str:>7}  {s_str:>6}  {v['r']:>+7.3f}  {effect_label(v['r'])}")

print(SEP)
print("""
  CATATAN
  • Wilcoxon two-sided, α=0.05. Dengan 5 fold, power uji rendah →
    perhatikan CI dan effect size sebagai bukti tambahan.
  • CI mencakup 0  → perbedaan TIDAK konsisten di semua fold.
  • Δ negatif = proposed sedikit di bawah baseline.
""")

  WILCOXON SIGNED-RANK TEST + BOOTSTRAP CI
  Proposed : RandomForest P=40% (76 fitur)
  Baseline : Original model
  Folds: 5  |  α = 0.05  |  Bootstrap CI: 95%  |  n_boot: 10.000

  Metrik  : Accuracy
----------------------------------------------------------------------
  Baseline mean ± std  : 0.9626 ± 0.0005
  Proposed mean ± std  : 0.9617 ± 0.0005
  Δ mean (Prop - Base) : -0.0009
  Bootstrap 95% CI (Δ) : [-0.0013,  -0.0005]  ← CI tidak mencakup 0 ✓
  Wilcoxon W           : 0.0
  p-value              : 0.0625
  Kesimpulan           : ✗ tidak signifikan
  Effect size r        : -1.0000  → besar

  Metrik  : F1 Macro
----------------------------------------------------------------------
  Baseline mean ± std  : 0.9618 ± 0.0006
  Proposed mean ± std  : 0.9609 ± 0.0006
  Δ mean (Prop - Base) : -0.0009
  Bootstrap 95% CI (Δ) : [-0.0013,  -0.0005]  ← CI tidak mencakup 0 ✓
  Wilcoxon W           : 0.0
  p-value              : 0.0625
  Kesimpulan           : ✗ tidak signifikan
  Effect size

## Dataset 3

In [2]:
"""
Analisis Statistik: Wilcoxon Signed-Rank Test + Confidence Interval
Perbandingan Best Scenario (LightGBM P=10%) vs Baseline (LightGBM 236 fitur)
"""

import numpy as np
from scipy import stats

# ─────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────

# Baseline LightGBM (236 fitur) — per fold
baseline = {
    "Accuracy":   np.array([0.9275, 0.9293, 0.9311, 0.9284, 0.9287]),
    "F1 Macro":   np.array([0.7140, 0.7272, 0.7723, 0.6810, 0.7336]),
    "F1 Weighted":np.array([0.9260, 0.9277, 0.9303, 0.9262, 0.9275]),
}

# Best Scenario: LightGBM P=10% (24 fitur) — per fold
best = {
    "Accuracy":   np.array([0.9221, 0.9155, 0.9263, 0.9134, 0.9152]),
    "F1 Macro":   np.array([0.7291, 0.7090, 0.7956, 0.6916, 0.7527]),
    "F1 Weighted":np.array([0.9212, 0.9143, 0.9262, 0.9126, 0.9154]),
}

METRICS = list(baseline.keys())
N_FOLDS = 5
ALPHA = 0.05

# ─────────────────────────────────────────────
# FUNGSI HELPER
# ─────────────────────────────────────────────

def bootstrap_ci(diff, n_boot=10_000, ci=95, seed=42):
    """Bootstrap CI untuk mean difference."""
    rng = np.random.default_rng(seed)
    means = [rng.choice(diff, size=len(diff), replace=True).mean()
             for _ in range(n_boot)]
    lo = np.percentile(means, (100 - ci) / 2)
    hi = np.percentile(means, 100 - (100 - ci) / 2)
    return lo, hi

def rank_biserial(diff):
    """Effect size r untuk Wilcoxon (rank-biserial correlation)."""
    pos = np.sum(diff[diff > 0])
    neg = np.abs(np.sum(diff[diff < 0]))
    return (pos - neg) / (pos + neg) if (pos + neg) > 0 else 0.0

def interpret_effect(r):
    r = abs(r)
    if r < 0.1:   return "sangat kecil"
    elif r < 0.3: return "kecil"
    elif r < 0.5: return "sedang"
    else:         return "besar"

def interpret_direction(mean_diff):
    if mean_diff > 0:  return "Best > Baseline"
    elif mean_diff < 0: return "Best < Baseline"
    else:               return "Tidak ada perbedaan"

# ─────────────────────────────────────────────
# ANALISIS UTAMA
# ─────────────────────────────────────────────

SEP  = "=" * 74
SEP2 = "-" * 74

print(SEP)
print("  ANALISIS STATISTIK: BEST SCENARIO vs BASELINE")
print("  Best   : LightGBM P=10% (24 fitur)")
print("  Baseline: LightGBM     (236 fitur)")
print(f"  Folds  : {N_FOLDS}  |  Alpha : {ALPHA}  |  Bootstrap CI: 95%")
print(SEP)

results = {}

for metric in METRICS:
    b = baseline[metric]
    s = best[metric]
    diff = s - b          # positif = best unggul

    mean_b    = b.mean()
    mean_s    = s.mean()
    mean_diff = diff.mean()
    std_diff  = diff.std(ddof=1)

    # Wilcoxon Signed-Rank Test
    try:
        stat, p_val = stats.wilcoxon(s, b, alternative="two-sided")
        test_note = ""
    except ValueError as e:
        stat, p_val = np.nan, np.nan
        test_note = f" [!] {e}"

    # 95% Bootstrap CI untuk mean difference
    ci_lo, ci_hi = bootstrap_ci(diff)

    # Effect size
    r = rank_biserial(diff)

    results[metric] = {
        "mean_baseline": mean_b,
        "mean_best":     mean_s,
        "mean_diff":     mean_diff,
        "std_diff":      std_diff,
        "stat":          stat,
        "p_val":         p_val,
        "ci_lo":         ci_lo,
        "ci_hi":         ci_hi,
        "r":             r,
        "significant":   p_val < ALPHA if not np.isnan(p_val) else False,
    }

    sig_mark = "* SIGNIFIKAN *" if results[metric]["significant"] else "tidak signifikan"
    direction = interpret_direction(mean_diff)
    effect    = interpret_effect(r)

    print(f"\n  Metrik  : {metric}")
    print(SEP2)
    print(f"  Baseline (mean ± std) : {mean_b:.4f} ± {b.std(ddof=1):.4f}")
    print(f"  Best     (mean ± std) : {mean_s:.4f} ± {s.std(ddof=1):.4f}")
    print(f"  Δ mean (Best - Base)  : {mean_diff:+.4f}  ({direction})")
    print(f"  Bootstrap 95% CI (Δ)  : [{ci_lo:+.4f}, {ci_hi:+.4f}]")
    print(f"  Wilcoxon W            : {stat:.1f}")
    print(f"  p-value               : {p_val:.4f}{test_note}")
    print(f"  Kesimpulan            : {sig_mark}")
    print(f"  Effect size (r)       : {r:+.4f}  → {effect}")

# ─────────────────────────────────────────────
# RINGKASAN TABEL
# ─────────────────────────────────────────────

print(f"\n\n{SEP}")
print("  RINGKASAN HASIL")
print(SEP)
print(f"  {'Metrik':<20} {'Δ Mean':>8} {'95% CI':>22} {'p-val':>8} {'Sig?':>6} {'r':>7} {'Interpretasi'}")
print(SEP2)

for metric, r in results.items():
    ci_str  = f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]"
    sig_str = "YA  ✓" if r["significant"] else "tidak"
    eff_str = interpret_effect(r["r"])
    p_str   = f"{r['p_val']:.4f}" if not np.isnan(r["p_val"]) else "  NaN"
    print(f"  {metric:<20} {r['mean_diff']:>+8.4f} {ci_str:>22} {p_str:>8} {sig_str:>6} {r['r']:>+7.3f}  {eff_str}")

print(SEP)
print("""
  CATATAN INTERPRETASI
  ─────────────────────────────────────────────────────────────────────
  • Wilcoxon Signed-Rank Test (two-sided, α=0.05) — non-parametrik,
    cocok untuk n_fold kecil (5 fold) tanpa asumsi normalitas.
  • Bootstrap CI dihitung dari 10.000 resample atas selisih per fold.
  • CI yang TIDAK mencakup 0 → perbedaan konsisten antar fold.
  • Effect size r (rank-biserial): |r|<0.1 kecil, 0.3 sedang, >0.5 besar.
  • Dengan hanya 5 fold, power uji rendah — p-value tinggi bukan
    berarti tidak ada perbedaan nyata; lihat CI dan effect size juga.
""")

  ANALISIS STATISTIK: BEST SCENARIO vs BASELINE
  Best   : LightGBM P=10% (24 fitur)
  Baseline: LightGBM     (236 fitur)
  Folds  : 5  |  Alpha : 0.05  |  Bootstrap CI: 95%

  Metrik  : Accuracy
--------------------------------------------------------------------------
  Baseline (mean ± std) : 0.9290 ± 0.0013
  Best     (mean ± std) : 0.9185 ± 0.0055
  Δ mean (Best - Base)  : -0.0105  (Best < Baseline)
  Bootstrap 95% CI (Δ)  : [-0.0142, -0.0067]
  Wilcoxon W            : 0.0
  p-value               : 0.0625
  Kesimpulan            : tidak signifikan
  Effect size (r)       : -1.0000  → besar

  Metrik  : F1 Macro
--------------------------------------------------------------------------
  Baseline (mean ± std) : 0.7256 ± 0.0331
  Best     (mean ± std) : 0.7356 ± 0.0406
  Δ mean (Best - Base)  : +0.0100  (Best > Baseline)
  Bootstrap 95% CI (Δ)  : [-0.0049, +0.0200]
  Wilcoxon W            : 3.0
  p-value               : 0.3125
  Kesimpulan            : tidak signifikan
  Effect size

## Dataset 4 16 class

In [5]:
"""
Analisis Statistik: Wilcoxon Signed-Rank Test + Bootstrap CI
Proposed: RandomForest P=40% (22 fitur) vs Baseline
Metrik: Accuracy, F1 Macro, F1 Weighted
"""

import numpy as np
from scipy import stats

baseline = {
    "Accuracy":    np.array([0.7484, 0.7543, 0.7536, 0.7458, 0.7525]),
    "F1 Macro":    np.array([0.5301, 0.5396, 0.5374, 0.5233, 0.5376]),
    "F1 Weighted": np.array([0.7465, 0.7523, 0.7507, 0.7437, 0.7507]),
}

proposed = {
    "Accuracy":    np.array([0.7464, 0.7426, 0.7511, 0.7375, 0.7468]),
    "F1 Macro":    np.array([0.5283, 0.5211, 0.5368, 0.5111, 0.5295]),
    "F1 Weighted": np.array([0.7460, 0.7424, 0.7499, 0.7368, 0.7462]),
}

METRICS = ["Accuracy", "F1 Macro", "F1 Weighted"]
ALPHA   = 0.05

def bootstrap_ci(diff, n_boot=10_000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    means = [rng.choice(diff, size=len(diff), replace=True).mean()
             for _ in range(n_boot)]
    return np.percentile(means, (100-ci)/2), np.percentile(means, 100-(100-ci)/2)

def rank_biserial(diff):
    pos = np.sum(diff[diff > 0])
    neg = abs(np.sum(diff[diff < 0]))
    return (pos - neg) / (pos + neg) if (pos + neg) > 0 else 0.0

def effect_label(r):
    r = abs(r)
    if r < 0.1:   return "sangat kecil"
    elif r < 0.3: return "kecil"
    elif r < 0.5: return "sedang"
    else:         return "besar"

SEP  = "=" * 70
SEP2 = "-" * 70

print(SEP)
print("  WILCOXON SIGNED-RANK TEST + BOOTSTRAP CI")
print("  Proposed : RandomForest P=40% (22 fitur)")
print("  Baseline : Original model")
print(f"  Folds: 5  |  α = {ALPHA}  |  Bootstrap CI: 95%  |  n_boot: 10.000")
print(SEP)

results = {}

for metric in METRICS:
    b    = baseline[metric]
    p    = proposed[metric]
    diff = p - b

    mean_b    = b.mean()
    mean_p    = p.mean()
    mean_diff = diff.mean()

    try:
        stat, p_val = stats.wilcoxon(p, b, alternative="two-sided")
    except ValueError:
        stat, p_val = np.nan, np.nan

    ci_lo, ci_hi = bootstrap_ci(diff)
    r = rank_biserial(diff)

    results[metric] = dict(mean_b=mean_b, mean_p=mean_p,
                           mean_diff=mean_diff, stat=stat,
                           p_val=p_val, ci_lo=ci_lo, ci_hi=ci_hi, r=r)

    sig  = (not np.isnan(p_val)) and p_val < ALPHA
    mark = "✓ SIGNIFIKAN" if sig else "✗ tidak signifikan"
    ci_covers_zero = ci_lo <= 0 <= ci_hi

    print(f"\n  Metrik  : {metric}")
    print(SEP2)
    print(f"  Baseline mean ± std  : {mean_b:.4f} ± {b.std(ddof=1):.4f}")
    print(f"  Proposed mean ± std  : {mean_p:.4f} ± {p.std(ddof=1):.4f}")
    print(f"  Δ mean (Prop - Base) : {mean_diff:+.4f}")
    print(f"  Bootstrap 95% CI (Δ) : [{ci_lo:+.4f},  {ci_hi:+.4f}]"
          + ("  ← CI mencakup 0" if ci_covers_zero else "  ← CI tidak mencakup 0"))
    print(f"  Wilcoxon W           : {stat:.1f}")
    print(f"  p-value              : {p_val:.4f}")
    print(f"  Kesimpulan           : {mark}")
    print(f"  Effect size r        : {r:+.4f}  → {effect_label(r)}")

print(f"\n\n{SEP}")
print("  TABEL RINGKASAN")
print(SEP)
print(f"  {'Metrik':<14} {'Base':>7} {'Prop':>7} {'Δ':>8}  {'95% CI Δ':>22}  {'p-val':>7}  {'Sig':>6}  {'r':>7}  Efek")
print(SEP2)

for m, v in results.items():
    ci_str = f"[{v['ci_lo']:+.4f}, {v['ci_hi']:+.4f}]"
    sig    = (not np.isnan(v['p_val'])) and v['p_val'] < ALPHA
    s_str  = "YA ✓" if sig else "tidak"
    p_str  = f"{v['p_val']:.4f}" if not np.isnan(v['p_val']) else "  NaN"
    print(f"  {m:<14} {v['mean_b']:>7.4f} {v['mean_p']:>7.4f} {v['mean_diff']:>+8.4f}  {ci_str:>22}  {p_str:>7}  {s_str:>6}  {v['r']:>+7.3f}  {effect_label(v['r'])}")

print(SEP)
print("""
  CATATAN
  • Wilcoxon two-sided, α=0.05. Dengan 5 fold, power uji rendah →
    p minimum yang bisa dicapai adalah 0.0625 (n=5).
  • CI mencakup 0  → perbedaan tidak konsisten di semua fold.
  • CI tidak mencakup 0 → perbedaan konsisten antar fold.
  • Δ negatif = proposed di bawah baseline.
""")

  WILCOXON SIGNED-RANK TEST + BOOTSTRAP CI
  Proposed : RandomForest P=40% (22 fitur)
  Baseline : Original model
  Folds: 5  |  α = 0.05  |  Bootstrap CI: 95%  |  n_boot: 10.000

  Metrik  : Accuracy
----------------------------------------------------------------------
  Baseline mean ± std  : 0.7509 ± 0.0037
  Proposed mean ± std  : 0.7449 ± 0.0051
  Δ mean (Prop - Base) : -0.0060
  Bootstrap 95% CI (Δ) : [-0.0092,  -0.0029]  ← CI tidak mencakup 0
  Wilcoxon W           : 0.0
  p-value              : 0.0625
  Kesimpulan           : ✗ tidak signifikan
  Effect size r        : -1.0000  → besar

  Metrik  : F1 Macro
----------------------------------------------------------------------
  Baseline mean ± std  : 0.5336 ± 0.0068
  Proposed mean ± std  : 0.5254 ± 0.0097
  Δ mean (Prop - Base) : -0.0082
  Bootstrap 95% CI (Δ) : [-0.0139,  -0.0026]  ← CI tidak mencakup 0
  Wilcoxon W           : 0.0
  p-value              : 0.0625
  Kesimpulan           : ✗ tidak signifikan
  Effect size r  

## Dataset 4 4 class

In [6]:
"""
Analisis Statistik: Wilcoxon Signed-Rank Test + Bootstrap CI
Proposed: RandomForest P=40% (22 fitur) vs Baseline RandomForest
Metrik: Accuracy, F1 Macro, F1 Weighted
"""

import numpy as np
from scipy import stats

baseline = {
    "Accuracy":    np.array([0.8757, 0.8734, 0.8753, 0.8726, 0.8708]),
    "F1 Macro":    np.array([0.8136, 0.8096, 0.8127, 0.8087, 0.8059]),
    "F1 Weighted": np.array([0.8759, 0.8733, 0.8753, 0.8726, 0.8709]),
}

proposed = {
    "Accuracy":    np.array([0.8767, 0.8755, 0.8804, 0.8772, 0.8765]),
    "F1 Macro":    np.array([0.8149, 0.8127, 0.8202, 0.8155, 0.8143]),
    "F1 Weighted": np.array([0.8768, 0.8753, 0.8804, 0.8771, 0.8765]),
}

METRICS = ["Accuracy", "F1 Macro", "F1 Weighted"]
ALPHA   = 0.05

def bootstrap_ci(diff, n_boot=10_000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    means = [rng.choice(diff, size=len(diff), replace=True).mean()
             for _ in range(n_boot)]
    return np.percentile(means, (100-ci)/2), np.percentile(means, 100-(100-ci)/2)

def rank_biserial(diff):
    pos = np.sum(diff[diff > 0])
    neg = abs(np.sum(diff[diff < 0]))
    return (pos - neg) / (pos + neg) if (pos + neg) > 0 else 0.0

def effect_label(r):
    r = abs(r)
    if r < 0.1:   return "sangat kecil"
    elif r < 0.3: return "kecil"
    elif r < 0.5: return "sedang"
    else:         return "besar"

SEP  = "=" * 70
SEP2 = "-" * 70

print(SEP)
print("  WILCOXON SIGNED-RANK TEST + BOOTSTRAP CI")
print("  Proposed : RandomForest P=40% (22 fitur)")
print("  Baseline : RandomForest (full fitur)")
print(f"  Folds: 5  |  α = {ALPHA}  |  Bootstrap CI: 95%  |  n_boot: 10.000")
print(SEP)

results = {}

for metric in METRICS:
    b    = baseline[metric]
    p    = proposed[metric]
    diff = p - b

    mean_b    = b.mean()
    mean_p    = p.mean()
    mean_diff = diff.mean()

    try:
        stat, p_val = stats.wilcoxon(p, b, alternative="two-sided")
    except ValueError:
        stat, p_val = np.nan, np.nan

    ci_lo, ci_hi = bootstrap_ci(diff)
    r = rank_biserial(diff)

    results[metric] = dict(mean_b=mean_b, mean_p=mean_p,
                           mean_diff=mean_diff, stat=stat,
                           p_val=p_val, ci_lo=ci_lo, ci_hi=ci_hi, r=r)

    sig  = (not np.isnan(p_val)) and p_val < ALPHA
    mark = "✓ SIGNIFIKAN" if sig else "✗ tidak signifikan"
    ci_covers_zero = ci_lo <= 0 <= ci_hi

    print(f"\n  Metrik  : {metric}")
    print(SEP2)
    print(f"  Baseline mean ± std  : {mean_b:.4f} ± {b.std(ddof=1):.4f}")
    print(f"  Proposed mean ± std  : {mean_p:.4f} ± {p.std(ddof=1):.4f}")
    print(f"  Δ mean (Prop - Base) : {mean_diff:+.4f}")
    print(f"  Bootstrap 95% CI (Δ) : [{ci_lo:+.4f},  {ci_hi:+.4f}]"
          + ("  ← CI mencakup 0" if ci_covers_zero else "  ← CI tidak mencakup 0 ✓"))
    print(f"  Wilcoxon W           : {stat:.1f}")
    print(f"  p-value              : {p_val:.4f}")
    print(f"  Kesimpulan           : {mark}")
    print(f"  Effect size r        : {r:+.4f}  → {effect_label(r)}")

print(f"\n\n{SEP}")
print("  TABEL RINGKASAN")
print(SEP)
print(f"  {'Metrik':<14} {'Base':>7} {'Prop':>7} {'Δ':>8}  {'95% CI Δ':>22}  {'p-val':>7}  {'Sig':>6}  {'r':>7}  Efek")
print(SEP2)

for m, v in results.items():
    ci_str = f"[{v['ci_lo']:+.4f}, {v['ci_hi']:+.4f}]"
    sig    = (not np.isnan(v['p_val'])) and v['p_val'] < ALPHA
    s_str  = "YA ✓" if sig else "tidak"
    p_str  = f"{v['p_val']:.4f}" if not np.isnan(v['p_val']) else "  NaN"
    print(f"  {m:<14} {v['mean_b']:>7.4f} {v['mean_p']:>7.4f} {v['mean_diff']:>+8.4f}  {ci_str:>22}  {p_str:>7}  {s_str:>6}  {v['r']:>+7.3f}  {effect_label(v['r'])}")

print(SEP)
print("""
  CATATAN
  • Wilcoxon two-sided, α=0.05. Dengan 5 fold, p minimum = 0.0625.
  • CI tidak mencakup 0 → perbedaan konsisten di semua fold.
  • |r| = 1.0 → proposed selalu unggul di seluruh fold tanpa pengecualian.
""")

  WILCOXON SIGNED-RANK TEST + BOOTSTRAP CI
  Proposed : RandomForest P=40% (22 fitur)
  Baseline : RandomForest (full fitur)
  Folds: 5  |  α = 0.05  |  Bootstrap CI: 95%  |  n_boot: 10.000

  Metrik  : Accuracy
----------------------------------------------------------------------
  Baseline mean ± std  : 0.8736 ± 0.0020
  Proposed mean ± std  : 0.8773 ± 0.0019
  Δ mean (Prop - Base) : +0.0037
  Bootstrap 95% CI (Δ) : [+0.0020,  +0.0052]  ← CI tidak mencakup 0 ✓
  Wilcoxon W           : 0.0
  p-value              : 0.0625
  Kesimpulan           : ✗ tidak signifikan
  Effect size r        : +1.0000  → besar

  Metrik  : F1 Macro
----------------------------------------------------------------------
  Baseline mean ± std  : 0.8101 ± 0.0031
  Proposed mean ± std  : 0.8155 ± 0.0028
  Δ mean (Prop - Base) : +0.0054
  Bootstrap 95% CI (Δ) : [+0.0029,  +0.0077]  ← CI tidak mencakup 0 ✓
  Wilcoxon W           : 0.0
  p-value              : 0.0625
  Kesimpulan           : ✗ tidak signifikan
  

## Dataset 5